
# Quick Model Tour — Legal Angle (Optional)

**Day 1 — AI Foundations · Practical 5 of 6 (light/optional) · Companion to the "Modern Model
Landscape" deck**

> **Running in Google Colab:** works fine on the default **CPU runtime**. A GPU runtime speeds
> up Part 2's SLM generation but isn't required for this short notebook.

---

## Learning Objectives

By the end of this short notebook, you will have **hands-on contact** with one model from each
of three size/purpose tiers discussed in the deck:

1. A **legal-domain encoder model** (LegalBERT) — classify a clause by type
2. A **small language model (SLM)** — generate a short legal disclaimer on-device-scale
3. A **conceptual note on VLMs** for scanned legal documents (not run — API/hardware dependent)

## Why This Matters for a Law Firm

The Model Landscape deck covers a lot of ground (LLMs, VLMs, SLMs, audio, embeddings). This
notebook isn't meant to be exhaustive — it's a **10-minute proof that these aren't just slide
bullet points**. By the time this notebook is done, trainees will have personally run a
legal-domain model and a small on-device-class model.

> **Time budget note:** this is the first notebook to skip if the day is running long — the
> deck's content stands on its own without this hands-on tour.

## Notebook Workflow

```mermaid
flowchart LR
    A["Clause text"] --> B["LegalBERT\n(legal-domain encoder)"]
    B --> C["Clause-type\nclassification"]

    D["Prompt"] --> E["Small SLM\n(e.g. Qwen2.5-0.5B)"]
    E --> F["Generated\nlegal disclaimer"]

    G["Scanned PDF\ncontract page"] -.->|"conceptual only,\nnot run here"| H["VLM\n(OCR + layout understanding)"]



## Part 1 — LegalBERT: A Legal-Domain Encoder Model

### Section 1 — Setup

We already met `nlpaueb/legal-bert-base-uncased` in the Tokenization notebook. Here we use its
full model (not just the tokenizer) with a **zero-shot-style classification approach**: we
compute sentence embeddings for a set of candidate clause-type labels and for our target
clause, then find the closest match by cosine similarity. This mirrors the LEDGAR benchmark's
task (classifying a contract provision into one of ~100 categories) at small scale.


In [ ]:

# Install dependencies (accelerate is required for device_map="auto" in Part 2).
# Running in Google Colab: this cell installs everything needed -- just run it.
%pip install -q transformers torch accelerate

import torch
from transformers import AutoTokenizer, AutoModel
import torch.nn.functional as F

MODEL_NAME = "nlpaueb/legal-bert-base-uncased"

legalbert_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
legalbert_model = AutoModel.from_pretrained(MODEL_NAME)
legalbert_model.eval()

print(f"Loaded {MODEL_NAME}")



### Section 2 — Embed Text with Mean Pooling

To turn a sentence into a single vector, we run it through the model and **mean-pool** the
token-level hidden states (averaging across all tokens, weighted by the attention mask so
padding tokens don't count). This is a standard, simple way to get a sentence embedding from an
encoder model like BERT.


In [ ]:

def embed_text(text):
    inputs = legalbert_tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = legalbert_model(**inputs)

    # outputs.last_hidden_state: (batch=1, seq_len, hidden_size)
    token_embeddings = outputs.last_hidden_state
    attention_mask = inputs["attention_mask"].unsqueeze(-1)  # (1, seq_len, 1)

    # Mean-pool over real tokens only (mask out padding)
    summed = (token_embeddings * attention_mask).sum(dim=1)
    counts = attention_mask.sum(dim=1).clamp(min=1e-9)
    mean_pooled = summed / counts

    return F.normalize(mean_pooled, p=2, dim=1)  # L2-normalize for cosine similarity



### Section 3 — Classify a Clause by Type

We define a small set of candidate clause-type labels (inspired by CUAD's 13 clause categories
and LEDGAR's provision-type taxonomy), embed each label description, embed our target clause,
and rank labels by cosine similarity.


In [ ]:

candidate_labels = {
    "Indemnification": "This clause requires one party to compensate the other for losses or damages.",
    "Termination": "This clause describes how and when the agreement may be ended.",
    "Confidentiality": "This clause restricts disclosure of confidential or proprietary information.",
    "Governing Law": "This clause specifies which jurisdiction's laws govern the agreement.",
    "Force Majeure": "This clause excuses performance due to events beyond the parties' control.",
    "Arbitration": "This clause requires disputes to be resolved through arbitration rather than litigation.",
}

target_clause = (
    "Either party may terminate this Agreement immediately upon written notice if the other "
    "party materially breaches any provision hereof and fails to cure such breach within "
    "fifteen (15) days."
)

target_embedding = embed_text(target_clause)

print(f"Target clause: {target_clause!r}\n")
print("Similarity to each candidate clause type:\n")

scores = {}
for label, description in candidate_labels.items():
    label_embedding = embed_text(description)
    similarity = (target_embedding @ label_embedding.T).item()
    scores[label] = similarity

for label, score in sorted(scores.items(), key=lambda kv: -kv[1]):
    print(f"  {label:<18s} {score:.4f}")

predicted_label = max(scores, key=scores.get)
print(f"\nPredicted clause type: {predicted_label}")



Try changing `target_clause` above to a different clause (an indemnification clause, a
confidentiality clause) and re-run — the predicted label should shift accordingly. This is a
simplified, embedding-similarity version of the kind of clause-classification system a firm
might build on top of a legal-domain encoder model.



---

## Part 2 — A Small Language Model (SLM) in Action

### Section 4 — Load and Run a Tiny SLM

Small Language Models are designed to run efficiently — on a single GPU, or even on-device —
while still being genuinely useful for constrained tasks. Here we load a small instruction-
following model and ask it to draft a short, generic legal disclaimer. This is illustrative of
the SLM tier discussed in the deck (Phi-4, Gemma 3, Qwen 2.5 family): fast, cheap, "good enough"
for narrow, well-defined tasks.

> If `Qwen/Qwen2.5-0.5B-Instruct` is slow to download in your environment, any small instruct
> model already cached locally works as a substitute — the point is the tier, not the exact
> checkpoint.


In [ ]:

from transformers import pipeline

SLM_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

slm_pipe = pipeline(
    "text-generation",
    model=SLM_MODEL_NAME,
    torch_dtype="auto",
    device_map="auto" if torch.cuda.is_available() else None,
)

print(f"Loaded SLM: {SLM_MODEL_NAME}")


In [ ]:

messages = [
    {
        "role": "user",
        "content": (
            "Draft a short, generic 'attorney advertising' disclaimer suitable for the "
            "footer of a law firm website. Keep it to 2-3 sentences."
        ),
    }
]

result = slm_pipe(messages, max_new_tokens=100, do_sample=True, temperature=0.7)

generated_text = result[0]["generated_text"][-1]["content"]
print("SLM-generated disclaimer:\n")
print(generated_text)



Notice how quickly this runs compared to a frontier closed model API call — that responsiveness
and low cost is exactly the tradeoff SLMs are built for: narrow tasks, high volume, tight
latency budgets (e.g. drafting boilerplate footer text across hundreds of documents), rather
than complex multi-step legal reasoning.



---

## Part 3 — VLMs for Scanned Legal Documents (Conceptual Note)

We don't run a VLM in this notebook (it typically requires either a GPU-hosted open model or an
API key, and image/PDF handling adds setup overhead not worth it for a 10-minute tour). But it's
worth stating explicitly where VLMs fit into a law firm's toolkit:

- **Scanned/photocopied contracts and exhibits** often arrive as images or low-quality PDFs,
  not clean text — a VLM can read the page layout *and* the text together, which is exactly
  what's needed for documents with signature blocks, stamps, handwritten annotations, or
  multi-column exhibit pages that a plain OCR pipeline handles poorly.
- **Redlines and mark-ups** — a VLM can visually distinguish inserted/deleted text formatting
  (strikethrough, underline, color) in a way that plain-text extraction loses entirely.
- This is the kind of task where a **closed frontier VLM** (Gemini 3 Pro, GPT-5 vision) or an
  **open-weight VLM** (Qwen3-VL) from the Modern Model Landscape deck would actually be used in
  production — worth a live demo separately if your firm works with scanned/paper documents
  regularly.



## Key Takeaways

1. **A legal-domain encoder model (LegalBERT)** can classify clause types using nothing more
   than sentence embeddings and cosine similarity — no finetuning required for this quick demo.
2. **Small language models** trade raw capability for speed and cost — genuinely useful for
   narrow, well-bounded tasks like boilerplate drafting, not for complex legal reasoning.
3. **VLMs** fill a real gap for law firms specifically: scanned documents, redlines, and
   annotated exhibits are common in legal practice and need vision, not just text, to handle
   correctly.
4. Picking the right tier (encoder classifier vs. SLM vs. frontier LLM vs. VLM) for a given
   task is a real engineering decision with cost and latency consequences — not just "always
   use the biggest model."

**Next up:** the *Legal Prompt Engineering Playground* — the main hands-on lab of Day 1.
